# Seminar 10: Score Matching

Notebook is based on https://github.com/acids-ircam/diffusion_models/.

Date: 2025-03-18

## Notebook Structure
1. **Introduction & Theoretical Recap**  
   We recall the key concepts behind score matching and its efficient variants.
2. **Score Matching & Langevin Dynamics**  
   We show how to learn the score function (i.e. gradients of log-density) using standard, sliced, and denoising score matching techniques and demonstrate sampling via Langevin dynamics.

## 0. Setup and Helper Functions

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

def hdr_plot_style():
    plt.style.use('dark_background')
    plt.rcParams.update({
        'font.size': 18,
        'lines.linewidth': 3,
        'lines.markersize': 15,
        'ps.useafm': True,
        'pdf.use14corefonts': True,
        'text.usetex': False,
        'font.family': 'sans-serif',
        'font.sans-serif': 'Courier New'
    })

hdr_plot_style()

# Set device
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

### Data Sampling and Visualization

We use the classic Swiss roll dataset (projected to 2D) for all our demonstrations. You can tweak the noise level via the `noise` parameter.

In [ ]:
def sample_batch(size, noise=0.5):
    # Note: Adjust noise level as needed.
    x, _ = make_swiss_roll(size, noise=noise)
    return x[:, [0, 2]] / 10.0

# Sample a dataset and plot it
data = sample_batch(10**4)
plt.figure(figsize=(16, 12))
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, color='red', edgecolor='white', s=40)
plt.title("Swiss Roll Data")
plt.show()

## 1. Score Matching and Langevin Dynamics

_Score matching_ aims to learn the _gradients_ (termed _score_) of $\log p(\mathbf{x})$ with respect to $\mathbf{x}$ instead of directly $\log p(\mathbf{x})$. Our goal is to make the score function of the model distribution as “close” as possible to the score function of the data distribution.
$$
\mathcal{F}_{\theta}(\mathbf{x}) \triangleq \nabla_x \log p_m(\mathbf{x} ; \theta) \approx \nabla_{\mathbf{x}} \log p(\mathbf{x})
$$

In particular, the objective in score matching is to minimize the Fisher divergence between these two score functions,

$$
\begin{aligned}
\hat{\theta}_{S M} & =\underset{\theta}{\arg \min } \, D_F\left(p, p_m\right) \\
& =\underset{\theta}{\arg \min } \, \frac{1}{2} \mathbb{E}_{\mathbf{x} \sim p(\mathbf{x})}\left[\left\|\nabla_\mathbf{x} \log p(\mathbf{x})-\nabla_\mathbf{x} \log p_m(\mathbf{x} ; \theta)\right\|_2^2\right].
\end{aligned}
$$

The following sections detail several formulations of score matching along with sampling schemes.

### 1.1 Standard Score Matching

We can rewrite the objective in the equation above so that it only depends on the unnormalized model density:

$$
\begin{align}
D_F\left(p_d, p_m\right) \propto L(\theta) &\triangleq \mathbb{E}_{\mathbf{x} \sim p(\mathbf{x})}\left[\operatorname{tr}\left(\nabla_\mathbf{x}^2 \log p_m(\mathbf{x} ; \theta)\right)+\frac{1}{2}\left\|\nabla_\mathbf{x} \log p_m(\mathbf{x} ; \theta)\right\|_2^2\right] \\
&= \mathbb{E}_{\mathbf{x} \sim p(\mathbf{x})} \left[ \text{ tr}\left( \nabla_{\mathbf{x}} \mathcal{F}_{\theta}(\mathbf{x})  \right) + \frac{1}{2} \left\Vert \mathcal{F}_{\theta}(\mathbf{x}) \right\lVert_2^2 \right],
\end{align}
$$

where $\nabla_{\mathbf{x}} \mathcal{F}_{\theta}(\mathbf{x})$ denotes the Jacobian of $\mathcal{F}_{\theta}(\mathbf{x})$ with respect to $\mathbf{x}$.

We can then approximate the objective with our data sample:

$$
L(\theta) \approx \frac{1}{n} \sum_{i=1}^n\left[\operatorname{tr}\left(\nabla_{\mathbf{x}} \mathcal{F}_{\theta}(\mathbf{x_i}) \right)+\frac{1}{2}\left\|\mathcal{F}_{\theta}(\mathbf{x_i})\right\|_2^2\right]
$$

In [ ]:
# Define a simple feedforward network for score approximation
model_score = nn.Sequential(
    nn.Linear(2, 128), nn.Softplus(),
    nn.Linear(128, 128), nn.Softplus(),
    nn.Linear(128, 2)
)

optimizer_score = optim.Adam(model_score.parameters(), lr=1e-3)

In [ ]:
def jacobian(f, x):
    """
    Compute the Jacobian of function f with respect to x.
    x: tensor of shape [B, N]
    Returns: tensor of shape [B, N, N]
    """
    B, N = x.shape
    y = f(x)
    jacobian_list = []
    for i in range(N):
        # Create one-hot vectors for each output dimension
        v = torch.zeros_like(y)
        v[:, i] = 1.
        dy_i_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True, create_graph=True)[0]
        jacobian_list.append(dy_i_dx)
    jacobian_mat = torch.stack(jacobian_list, dim=2)
    return jacobian_mat

In [ ]:
def score_matching_loss(model, samples):
    """
    Computes the standard score matching loss:
      L = trace(Jacobian) + 0.5 * ||f(x)||^2
    """
    samples.requires_grad_(True)
    fx = model(samples)
    # Compute the norm loss
    norm_loss = (torch.norm(fx, dim=-1) ** 2) / 2.
    # Compute the Jacobian loss
    jacobian_mat = jacobian(model, samples)
    trace_loss = torch.diagonal(jacobian_mat, dim1=-2, dim2=-1).sum(-1)
    return (trace_loss + norm_loss).mean()

#### Training the Score Matching Model

We train on the Swiss roll data.

In [ ]:
dataset = torch.tensor(data).float()
for t in range(2000):
    loss = score_matching_loss(model_score, dataset)
    optimizer_score.zero_grad()
    loss.backward()
    optimizer_score.step()
    if t % 500 == 0:
        print(f"Iteration {t} - Loss: {loss.item():.4f}")

### 1.2 Visualizing Learned Gradients

We plot the score function over the input space along with the data scatter. For improved numerical stability, we normalize the gradients for visualization.

In [ ]:
def plot_gradients(model, data_np, ax=None, plot_scatter=True):
    # Create a grid over the input space
    grid = np.stack(np.meshgrid(np.linspace(-1.5, 2.0, 50),
                                np.linspace(-1.5, 2.0, 50)), axis=-1).reshape(-1, 2)
    grid_tensor = torch.tensor(grid, dtype=torch.float32)
    with torch.no_grad():
        scores = model(grid_tensor).cpu().numpy()
    # Normalize scores for visualization
    scores_norm = np.linalg.norm(scores, axis=-1, keepdims=True) + 1e-9
    scores_log1p = scores / scores_norm * np.log1p(scores_norm)
    
    # Use existing axes if provided, else create a new figure
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 12))
    
    if plot_scatter:
        ax.scatter(data_np[:, 0], data_np[:, 1], alpha=0.3, color='red', edgecolor='white', s=40)
    ax.quiver(grid[:, 0], grid[:, 1], scores_log1p[:, 0], scores_log1p[:, 1], width=0.002, color='white')
    ax.set_xlim(-1.5, 2.0)
    ax.set_ylim(-1.5, 2.0)
    ax.set_title("Learned Score Function")
    
plot_gradients(model_score, data)

### 1.3 Langevin Dynamics Sampling

After training, our model is able to produce an approximation of the gradient of the probabiliity, such that $\mathcal{F}_{\theta}(\mathbf{x}) \approx \nabla_\mathbf{x} \log p(\mathbf{x})$. Therefore, we could use this to generate data by relying on a simple gradient ascent from a given point by using an initial sample $\mathbf{x}_{0} \sim \mathcal{N}(\mathbf{0},\mathbf{I})$, and then using the gradient information to find a local maximum of $p(\mathbf{x})$

$$\mathbf{x}_{t + 1} = \mathbf{x}_t + \epsilon \nabla_{\mathbf{x}_t} \log p(\mathbf{x}_t)$$

where $\epsilon$ defines the size of the step we take in the direction of the gradient (akin to the _learning rate_).

In [ ]:
def sample_simple(model, x, n_steps=20, eps=1e-3):
    x_seq = [x.unsqueeze(0)]
    for _ in range(n_steps):
        x = x + eps * model(x)
        x_seq.append(x.unsqueeze(0))
    return torch.cat(x_seq)

However, the previous procedure does not produce a true sample from $\mathbf{x} \sim p(\mathbf{x})$. In order to obtain such a sample, we can rely on a special case of _Langevin dynamics_. In this case, _Langevin dynamics_ can produce true samples from the density $p(\mathbf{x})$, by relying only on $\nabla_{\mathbf{x}} \log p(\mathbf{x})$. The sampling is defined in a way very similar to MCMC approaches, by applying recursively

$$\mathbf{x}_{t + 1} = \mathbf{x}_t + \frac{\epsilon}{2} \nabla_{\mathbf{x}_t} \log p(\mathbf{x}_t) + \sqrt{\epsilon} \mathbf{z}_{t}$$

where $\mathbf{z}_{t}\sim \mathcal{N}(\mathbf{0},\mathbf{I})$. It has been shown in [Welling et al. (2011)](https://www.ics.uci.edu/~welling/publications/papers/stoclangevin_v6.pdf) that under $\epsilon \rightarrow 0, t \rightarrow \inf$: $\mathbf{x}_t$ converges to an exact sample from $p(\mathbf{x})$. This is a key idea behind the _score-based generative modeling_ approach.

In order to implement this sampling procedure, we can once again start from $\mathbf{x}_{0} \sim \mathcal{N}(\mathbf{0},\mathbf{I})$, and progressively anneal $\epsilon \rightarrow 0$ at each step, to obtain true samples from $p(\mathbf{x})$.

In [ ]:
def sample_langevin(model, x, n_steps=10, eps=1e-2, decay=0.9, temperature=1.0):
    x_seq = [x.unsqueeze(0)]
    for _ in range(n_steps):
        z_t = torch.randn_like(x)
        x = x + (eps / 2) * model(x) + (np.sqrt(eps) * temperature * z_t)
        x_seq.append(x.unsqueeze(0))
        eps *= decay
    return torch.cat(x_seq)

In [ ]:
# Starting point
x0 = torch.tensor([1.5, -1.5])
samples_simple = sample_simple(model_score, x0)
samples_langevin = sample_langevin(model_score, x0)

# Plot Simple and Langevin sampling results side by side
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# Plot for Simple Sampling
plt.sca(axs[0])
plot_gradients(model_score, data, ax=axs[0])
axs[0].scatter(samples_simple[:, 0].detach().cpu(), samples_simple[:, 1].detach().cpu(),
               color='blue', edgecolor='white', s=150)

# Draw arrows for steps
deltas_simple = (samples_simple[1:] - samples_simple[:-1]).detach().cpu().numpy()
for i, delta in enumerate(deltas_simple):
    norm = np.linalg.norm(delta) + 1e-9
    adjusted = delta - delta / norm * 0.04
    axs[0].arrow(samples_simple[i, 0].item(), samples_simple[i, 1].item(),
                 adjusted[0], adjusted[1],
                 width=1e-4, head_width=2e-2, color="blue", linewidth=3)
axs[0].set_title("Simple Gradient Sampling")

# Plot for Langevin Dynamics Sampling
plt.sca(axs[1])
plot_gradients(model_score, data, ax=axs[1])
axs[1].scatter(samples_langevin[:, 0].detach().cpu(), samples_langevin[:, 1].detach().cpu(),
               color='green', edgecolor='white', s=150)

# Draw arrows for steps
deltas_langevin = (samples_langevin[1:] - samples_langevin[:-1]).detach().cpu().numpy()
for i, delta in enumerate(deltas_langevin):
    norm = np.linalg.norm(delta) + 1e-9
    adjusted = delta - delta / norm * 0.04
    axs[1].arrow(samples_langevin[i, 0].item(), samples_langevin[i, 1].item(),
                 adjusted[0], adjusted[1],
                 width=1e-4, head_width=2e-2, color="green", linewidth=3)
axs[1].set_title("Langevin Dynamics Sampling")

plt.tight_layout()
plt.show()

### 1.4 Sliced and Denoising Score Matching

The previously defined _score matching_ is not scalable to high-dimensional data, nor deep networks, because of the computation of $\text{ tr}\left( \nabla_{\mathbf{x}}  \mathcal{F}_{\theta}(\mathbf{x})  \right)$. Indeed, the computation of the Jacobian is a $O(N^2 + N)$ operation, thus not being suitable for high-dimensional problems.

#### 1.4.1 Sliced Score Matching

**Sliced Score Matching (SSM)** reduces computational complexity by projecting the gradients onto random vectors $\mathbf{v}$ drawn from a distribution (typically standard normal):

$$
\begin{aligned}
L_{\text{SSM}}(\theta) &\triangleq \mathbb{E}_{\mathbf{x}\sim p(\mathbf{x}),\, \mathbf{v}\sim \mathcal{N}(0,I)}\left[\mathbf{v}^\top \nabla_\mathbf{x}\mathcal{F}_\theta(\mathbf{x})\,\mathbf{v} + \frac{1}{2}\left(\mathbf{v}^\top \mathcal{F}_\theta(\mathbf{x})\right)^2\right]
\end{aligned}
$$

Empirically approximating with samples $\{\mathbf{x}_i\}_{i=1}^n$ and random projections $\{\mathbf{v}_i\}_{i=1}^n$:

$$
L_{\text{SSM}}(\theta) \approx \frac{1}{n}\sum_{i=1}^{n}\left[\mathbf{v}_i^\top \nabla_{\mathbf{x}}\mathcal{F}_\theta(\mathbf{x}_i)\,\mathbf{v}_i + \frac{1}{2}\left(\mathbf{v}_i^\top\mathcal{F}_\theta(\mathbf{x}_i)\right)^2\right], \quad \mathbf{v}_i \sim \mathcal{N}(0,I).
$$

This formulation significantly reduces computational costs, as it avoids computing the full Jacobian matrix.

In [ ]:
def sliced_score_matching(model, samples):
    samples.requires_grad_(True)
    # Construct random vectors
    vectors = torch.randn_like(samples)
    vectors = vectors / torch.norm(vectors, dim=-1, keepdim=True)
    # Compute the optimized vector-product jacobian
    logp, jvp = autograd.functional.jvp(model, samples, vectors, create_graph=True)
    # Compute the norm loss
    norm_loss = (logp * vectors) ** 2 / 2.
    # Compute the Jacobian loss
    v_jvp = jvp * vectors
    jacob_loss = v_jvp
    loss = jacob_loss + norm_loss
    return loss.mean(-1).mean(-1)

In [ ]:
# Our approximation model
model_score = nn.Sequential(
    nn.Linear(2, 128), nn.Softplus(),
    nn.Linear(128, 128), nn.Softplus(),
    nn.Linear(128, 2)
)
# Create ADAM optimizer over our model
optimizer = optim.Adam(model_score.parameters(), lr=1e-3)
dataset = torch.tensor(data)[:1000].float()
for t in range(2000):
    # Compute the loss.
    loss = sliced_score_matching(model_score, dataset)
    # Before the backward pass, zero all of the network gradients
    optimizer.zero_grad()
    # Backward pass: compute gradient of the loss with respect to parameters
    loss.backward()
    # Calling the step function to update the parameters
    optimizer.step()
    # Print loss
    if t % 500 == 0:
        print(f"Iteration {t} - Loss: {loss.item():.4f}")

In [ ]:
plot_gradients(model_score, data)

#### 1.4.2 Denoising Score Matching

**Denoising Score Matching (DSM)** is a variant of score matching that reformulates the original objective to avoid computationally expensive second-order derivatives. DSM introduces a conditional perturbation kernel $q(\tilde{\mathbf{x}}|\mathbf{x})$, which injects controlled noise into data points. The general DSM objective minimizes the discrepancy between the score function of the perturbed data distribution and the model's predicted score function:

$$
L_{\text{DSM}}(\theta) \coloneqq \frac{1}{2} \mathbb{E}_{\mathbf{x}\sim p(\mathbf{x}),\,\tilde{\mathbf{x}}\sim q(\tilde{\mathbf{x}}|\mathbf{x})}\left[\left\|\mathcal{F}_\theta(\tilde{\mathbf{x}})-\nabla_{\tilde{\mathbf{x}}}\log q(\tilde{\mathbf{x}}|\mathbf{x})\right\|_2^2\right],
$$

where $q(\tilde{\mathbf{x}}|\mathbf{x})$ is a user-defined noise-perturbation distribution that conditions on the original data point $\mathbf{x}$.

A particularly popular and practical choice for this noise distribution is the isotropic Gaussian:
$$
q_{\sigma}(\tilde{\mathbf{x}}|\mathbf{x}) = \mathcal{N}(\tilde{\mathbf{x}};\mathbf{x}, \sigma^2 I),\quad\sigma>0.
$$

With Gaussian noise, the DSM objective simplifies to:
$$
L_{\text{DSM}}(\theta) = \frac{1}{2}\mathbb{E}_{\mathbf{x}\sim p(\mathbf{x}),\,\epsilon\sim \mathcal{N}(0,I)}\left[\left\|\mathcal{F}_\theta(\mathbf{x}+\sigma \epsilon)+\frac{\epsilon}{\sigma}\right\|_2^2\right].
$$

Empirically, given a finite dataset $\{\mathbf{x}_i\}_{i=1}^n$, we approximate:
$$
L_{\text{DSM}}(\theta)\approx \frac{1}{2n}\sum_{i=1}^{n}\left\|\mathcal{F}_\theta(\mathbf{x}_i+\sigma\epsilon_i)+\frac{\epsilon_i}{\sigma}\right\|_2^2,\quad\epsilon_i\sim \mathcal{N}(0,I).
$$

DSM provides computational efficiency and scalability, avoiding explicit second-order derivatives, which is crucial in high-dimensional generative modeling.


In [ ]:
def denoising_score_matching_loss(model, samples, sigma=0.01):
    perturbed = samples + torch.randn_like(samples) * sigma
    target = - (perturbed - samples) / (sigma ** 2)
    scores = model(perturbed)
    # Flatten for loss computation
    target = target.view(target.shape[0], -1)
    scores = scores.view(scores.shape[0], -1)
    loss = 0.5 * ((scores - target) ** 2).sum(dim=-1).mean()
    return loss

In [ ]:
# Our approximation model
model_score = nn.Sequential(
    nn.Linear(2, 128), nn.Softplus(),
    nn.Linear(128, 128), nn.Softplus(),
    nn.Linear(128, 2)
)
# Create ADAM optimizer over our model
optimizer = optim.Adam(model_score.parameters(), lr=1e-3)
dataset = torch.tensor(data).float()
for t in range(5000):
    # Compute the loss.
    loss = denoising_score_matching_loss(model_score, dataset)
    # Before the backward pass, zero all of the network gradients
    optimizer.zero_grad()
    # Backward pass: compute gradient of the loss with respect to parameters
    loss.backward()
    # Calling the step function to update the parameters
    optimizer.step()
    # Print loss
    if t % 1000 == 0:
        print(f"Iteration {t} - Loss: {loss.item():.4f}")

In [ ]:
plot_gradients(model_score, data)

#### Noise-Conditional Score Networks (NCSN)

When modeling complex, high-dimensional data distributions, the data often lie near lower-dimensional manifolds embedded in the high-dimensional space, causing instability and numerical challenges during score estimation. To alleviate this issue, one can condition the score function explicitly on different noise scales, resulting in a family of noise-conditional score networks.

Formally, we define a noise-conditional score network as a score estimator parameterized explicitly by a noise-level parameter $\sigma$:
$$
\mathcal{F}_\theta(\mathbf{x}, \sigma) \approx \nabla_{\mathbf{x}}\log q_{\sigma}(\mathbf{x}), \quad \sigma > 0.
$$

Here, $q_{\sigma}(\mathbf{x})$ represents a noisy version of the data distribution defined as:
$$
q_{\sigma}(\mathbf{x})=\int p(\mathbf{x}_0)\,q_{\sigma}(\mathbf{x}|\mathbf{x}_0)\,d\mathbf{x}_0,
$$
where $q_{\sigma}(\mathbf{x}|\mathbf{x}_0)$ is a noise kernel that perturbs data points with controlled noise.

A practical implementation involves choosing a finite set of noise scales $\{\sigma_1,\dots,\sigma_L\}$, and conditioning the model explicitly on these discrete noise-level indices. The model thus becomes:
$$
\mathcal{F}_\theta(\mathbf{x}, \sigma_\ell),\quad \text{with}\quad \sigma_\ell\in\{\sigma_1,\dots,\sigma_L\}.
$$

To train such models efficiently, we define the annealed denoising score matching (DSM) loss, which weighs the contributions of different noise scales using an annealing schedule to ensure stable training across all scales:
$$
L_{\text{NCSN}}(\theta)=\frac{1}{2}\mathbb{E}_{\mathbf{x}\sim p(\mathbf{x}),\,\sigma_\ell\sim p(\sigma),\,\epsilon\sim\mathcal{N}(0,I)}\left[\sigma_\ell^{2}\left\|\mathcal{F}_\theta(\mathbf{x}+\sigma_\ell\epsilon,\sigma_\ell)+\frac{\epsilon}{\sigma_\ell}\right\|_2^2\right].
$$

Empirically, with data samples $\{\mathbf{x}_i\}_{i=1}^n$, the loss is approximated as:
$$
L_{\text{NCSN}}(\theta)\approx\frac{1}{2n}\sum_{i=1}^{n}\sigma_{y_i}^2\left\|\mathcal{F}_\theta(\mathbf{x}_i+\sigma_{y_i}\epsilon_i,\sigma_{y_i})+\frac{\epsilon_i}{\sigma_{y_i}}\right\|_2^2,\quad \epsilon_i\sim \mathcal{N}(0,I),
$$

where each $y_i$ is a randomly chosen index corresponding to a noise-level $\sigma_{y_i}$.

This approach significantly improves numerical stability, generalization, and quality of learned scores, enabling effective sampling from low-dimensional manifolds embedded within high-dimensional ambient spaces.

In [ ]:
class ConditionalLinear(nn.Module):
    def __init__(self, in_features, out_features, num_classes):
        super().__init__()
        self.lin = nn.Linear(in_features, out_features)
        self.embed = nn.Embedding(num_classes, out_features)
        # Use a standard initialization
        nn.init.uniform_(self.embed.weight, a=-0.1, b=0.1)

    def forward(self, x, y):
        out = self.lin(x)
        gamma = self.embed(y)
        return out + gamma

class ConditionalScoreModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.lin1 = ConditionalLinear(2, 128, num_classes)
        self.lin2 = ConditionalLinear(128, 128, num_classes)
        self.lin3 = nn.Linear(128, 2)
    
    def forward(self, x, y):
        x = F.softplus(self.lin1(x, y))
        x = F.softplus(self.lin2(x, y))
        return self.lin3(x)

In [ ]:
def anneal_dsm_loss(model, samples, labels, sigmas, anneal_power=2.0):
    used_sigmas = sigmas[labels].view(samples.shape[0], *([1] * (samples.ndimension() - 1)))
    perturbed = samples + torch.randn_like(samples) * used_sigmas
    target = -(perturbed - samples) / (used_sigmas ** 2)
    scores = model(perturbed, labels)
    target = target.view(target.shape[0], -1)
    scores = scores.view(scores.shape[0], -1)
    loss = 0.5 * ((scores - target) ** 2).sum(dim=-1) * (used_sigmas.view(-1) ** anneal_power)
    return loss.mean()

In [ ]:
# Set noise levels (geometric progression)
sigma_begin = 1.0
sigma_end = 0.01
num_classes = 4
sigmas = torch.tensor(np.exp(np.linspace(np.log(sigma_begin), np.log(sigma_end), num_classes))).float()

# Create a conditional score network and optimizer
model_cond = ConditionalScoreModel(num_classes)
optimizer_cond = optim.Adam(model_cond.parameters(), lr=1e-3)
dataset = torch.tensor(data).float()

for t in range(5000):
    labels = torch.randint(0, num_classes, (dataset.shape[0],))
    loss = anneal_dsm_loss(model_cond, dataset, labels, sigmas)
    optimizer_cond.zero_grad()
    loss.backward()
    optimizer_cond.step()
    if t % 500 == 0:
        print(f"Conditional DSM Iter {t} - Loss: {loss.item():.4f}")

In [ ]:
num_classes = len(sigmas)

fig, axs = plt.subplots(1, num_classes, figsize=(5*num_classes, 5))

grid = np.stack(np.meshgrid(np.linspace(-1.5, 2.0, 50),
                            np.linspace(-1.5, 2.0, 50)), axis=-1).reshape(-1, 2)

grid_tensor = torch.tensor(grid).float()

for label in range(num_classes):
    with torch.no_grad():
        labels_tensor = torch.full((grid.shape[0],), fill_value=label, dtype=torch.long)
        scores_cond = model_cond(torch.tensor(grid).float(), labels_tensor)
    
    scores_cond_np = scores_cond.cpu().numpy()
    scores_norm = np.linalg.norm(scores_cond_np, axis=-1, keepdims=True) + 1e-9
    scores_log1p = scores_cond_np / scores_norm * np.log1p(scores_norm)
    
    ax = axs[label]
    ax = axs[label]
    ax.scatter(data[:, 0], data[:, 1], alpha=0.3, color='red', edgecolor='white', s=40)
    ax.quiver(grid[:, 0], grid[:, 1], scores_log1p[:, 0], scores_log1p[:, 1],
              width=0.002, color='white')
    ax.set_title(f"CSF(Label={label}, σ={sigmas[label]:.3f})")
    ax.set_xlim(-1.5, 2.0)
    ax.set_ylim(-1.5, 2.0)

plt.tight_layout()
plt.show()

In [ ]:
# Create a grid of points covering the input space
grid_points = np.stack(
    np.meshgrid(
        np.linspace(-1.5, 2.0, 50),  # x-axis range
        np.linspace(-1.5, 2.0, 50)   # y-axis range
    ),
    axis=-1
).reshape(-1, 2)

# Assign a random noise-level label (from available sigmas) to each grid point
# This simulates drawing from a mixture of noise scales for the conditional score network.
random_labels = torch.randint(0, len(sigmas), (grid_points.shape[0],))

# Compute the conditional scores for each grid point using the trained conditional score network.
# The network takes both the grid point and its corresponding noise label as input.
scores = model_cond(torch.tensor(grid_points).float(), random_labels).detach()

# Normalize the scores for visualization:
# 1. Compute the L2 norm of each score vector.
# 2. Use a log1p transformation on the norm to scale the vectors,
#    which helps in enhancing visibility for vectors with small magnitudes.
scores_norm = np.linalg.norm(scores, axis=-1, ord=2, keepdims=True)
scores_log1p = scores / (scores_norm + 1e-9) * np.log1p(scores_norm)

# Plot the Swiss roll data and overlay the score field as a quiver plot.
plt.figure(figsize=(16, 12))
plt.scatter(data[:, 0], data[:, 1], alpha=0.3, color='red', edgecolor='white', s=40)
plt.quiver(
    grid_points[:, 0], grid_points[:, 1],
    scores_log1p[:, 0], scores_log1p[:, 1],
    width=0.002, color='white'
)
plt.xlim(-1.5, 2.0)
plt.ylim(-1.5, 2.0)
plt.title("Composite Conditional Score Field with Random Noise Labels")
plt.show()